# PlayReactant — Benchmark Sweep on Google Colab TPU/GPU

Resolution sweep for all 4 physics solvers (Stokes 2D/3D, PorowavesStokes 2D/3D)
using [Reactant.jl](https://github.com/EnzymeAD/Reactant.jl).

**Before running:** set *Runtime → Change runtime type* to **TPU v2-8** or **GPU (T4)**.

The backend is detected automatically when `BENCH_BACKEND=auto` (TPU → GPU → CPU).
Results are saved as a TOML file and can be plotted locally.

In [ ]:
# ── 1. Clone / update the PlayReactant repo ──────────────────────────
REPO_URL = "https://github.com/lraess/PlayReactant.git"  # ← update if needed
REPO_DIR = "/content/PlayReactant"
BRANCH   = "lr/bench"

if !isdir(REPO_DIR)
    run(`git clone --branch $BRANCH $REPO_URL $REPO_DIR`)
else
    run(`git -C $REPO_DIR fetch origin`)
    run(`git -C $REPO_DIR checkout $BRANCH`)
    run(`git -C $REPO_DIR pull`)
end
println("Repo ready at $REPO_DIR (branch: $BRANCH) ✓")

In [ ]:
# ── 2. Verify key scripts are present ────────────────────────────────
REPO_DIR = "/content/PlayReactant"
for s in ["Stokes_react2D.jl", "Stokes_react3D.jl",
          "PorowavesStokes_react2D.jl", "PorowavesStokes_react3D.jl",
          "bench_sweep.jl"]
    @assert isfile(joinpath(REPO_DIR, "egu26", s)) "Missing: $s"
end
println("All scripts found ✓")

In [ ]:
# ── 2. Configure and run the sweep ───────────────────────────────────
# Backend is detected automatically by the scripts (TPU → GPU → CPU) when set to "auto".
ENV["BENCH_BACKEND"]   = "auto"
ENV["BENCH_VIZ"]       = "0"
ENV["BENCH_SWEEP_OUT"] = "/content/bench_sweep_results.toml"
ENV["BENCH_RES_S2D"]   = "64,128,256,512"
ENV["BENCH_RES_S3D"]   = "32,64,128"
ENV["BENCH_RES_PW2D"]  = "32,64,128,256"
ENV["BENCH_RES_PW3D"]  = "16,32,64"

cd(joinpath("/content/PlayReactant", "egu26")) do
    include("bench_sweep.jl")
end

In [ ]:
# ── 4. Print results table ───────────────────────────────────────────
using TOML, Printf
data = TOML.parsefile("/content/bench_sweep_results.toml")
runs = data["runs"]
println("Backend: ", data["backend"], "\n")
@printf "%-20s %6s %12s %10s %8s %10s\n" "solver" "nx" "t_compile[s]" "t_run[s]" "niter" "T_eff[GB/s]"
println("-"^70)
for r in runs
    @printf "%-20s %6d %12.2f %10.3f %8d %10.2f\n" r["solver"] r["nx"] r["t_compile"] r["t_run"] r["niter"] r["T_eff"]
end

> **Remember to download your results!**
> Go to the Colab file browser (📁 left sidebar), navigate to `/content/`, right-click `bench_sweep_results.toml` and choose **Download**.